# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/clever-dhruv/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

print(os.getcwd())
print(os.listdir("/content"))


/content
['.config', 'flyrank-ml-internship', 'sample_data']


In [19]:
!git clone https://github.com/clever-dhruv/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [20]:
import pandas as pd

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.head())

Rows: 30000
Columns: 44
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ..

In [21]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())
print("Base rate:", df["is_declining_label"].mean())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Base rate: 0.5420666666666667


In [22]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, float("inf")],
    labels=["0-90", "91-180", "181-365", "365+"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)["is_declining_label"]
      .agg(
          decline_rate="mean",
          n="count"
      )
      .reset_index()
)

staleness_check["decline_rate"] = (
    staleness_check["decline_rate"] * 100
).round(2)

print(staleness_check)

  staleness_bucket  decline_rate      n
0             0-90         51.20  20655
1           91-180         61.11   9171
2          181-365         46.75    169
3             365+         60.00      5


In [23]:
### Signal 1 — Staleness

"""**Signal:** `days_since_last_update`

**Verdict: MIXED**

The bucket table shows some evidence that stale content is associated with decline. Pages
updated 91–180 days ago have a 61.11% decline rate compared with 51.20% for pages updated
within 90 days. However, the relationship is not monotonic: the 181–365 day bucket has a
46.75% decline rate. That bucket is also small (n=169), while the 365+ bucket has only n=5,
so those estimates are unstable. I therefore treat staleness as a mixed signal rather than
a reliable standalone predictor."""

'**Signal:** `days_since_last_update`\n\n**Verdict: MIXED**\n\nThe bucket table shows some evidence that stale content is associated with decline. Pages\nupdated 91–180 days ago have a 61.11% decline rate compared with 51.20% for pages updated\nwithin 90 days. However, the relationship is not monotonic: the 181–365 day bucket has a\n46.75% decline rate. That bucket is also small (n=169), while the 365+ bucket has only n=5,\nso those estimates are unstable. I therefore treat staleness as a mixed signal rather than\na reliable standalone predictor.'

In [24]:
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 299, 2999, 29999, float("inf")],
    labels=["1-299", "300-2,999", "3,000-29,999", "30,000+"]
)

volume_check = (
    df.groupby("volume_bucket", observed=False)["is_declining_label"]
      .agg(
          decline_rate="mean",
          n="count"
      )
      .reset_index()
)

volume_check["decline_rate"] = (
    volume_check["decline_rate"] * 100
).round(2)

print(volume_check)

  volume_bucket  decline_rate      n
0         1-299         45.39  11248
1     300-2,999         61.47  10469
2  3,000-29,999         58.61   7205
3       30,000+         46.20   1078


In [25]:
### Signal 2 — Search visibility

"""**Signal:** `impressions_90d`

**Verdict: MIXED**

Search visibility shows a non-linear relationship with decline. Pages with 300–2,999
impressions have a 61.47% decline rate, while pages with 3,000–29,999 impressions have a
58.61% decline rate. In contrast, low-volume pages have a 45.39% decline rate and the
30,000+ group has a 46.20% decline rate. This suggests that moderate search visibility may
represent a useful prioritization zone, but higher volume does not consistently mean higher
decline risk. I therefore treat search visibility as a mixed signal rather than a standalone
predictor."""

'**Signal:** `impressions_90d`\n\n**Verdict: MIXED**\n\nSearch visibility shows a non-linear relationship with decline. Pages with 300–2,999\nimpressions have a 61.47% decline rate, while pages with 3,000–29,999 impressions have a\n58.61% decline rate. In contrast, low-volume pages have a 45.39% decline rate and the\n30,000+ group has a 46.20% decline rate. This suggests that moderate search visibility may\nrepresent a useful prioritization zone, but higher volume does not consistently mean higher\ndecline risk. I therefore treat search visibility as a mixed signal rather than a standalone\npredictor.'

In [26]:
stale = (
    df["days_since_last_update"].between(91, 365)
).astype(int)

visible = (
    df["impressions_90d"].between(300, 29999)
).astype(int)

df["score"] = stale * visible * df["impressions_90d"]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
### Baseline rule

"""Prioritize pages that have not been updated for 91–365 days and have
300–29,999 search impressions over the trailing 90 days.

The score is the page's 90-day impressions when both conditions are met,
and zero otherwise. This makes higher-visibility pages within the observed
opportunity zone rank earlier.

Reason code: `stale_and_visible`

Action label: `refresh_review`"""

"Prioritize pages that have not been updated for 91–365 days and have\n300–29,999 search impressions over the trailing 90 days.\n\nThe score is the page's 90-day impressions when both conditions are met,\nand zero otherwise. This makes higher-visibility pages within the observed\nopportunity zone rank earlier.\n\nReason code: `stale_and_visible`\n\nAction label: `refresh_review`"

In [28]:
# Section 2 — Build the baseline ranked queue

stale = (
    df["days_since_last_update"].between(91, 365)
).astype(int)

visible = (
    df["impressions_90d"].between(300, 29999)
).astype(int)

df["score"] = stale * visible * df["impressions_90d"]

df["reason_code"] = np.where(
    df["score"] > 0,
    "stale_and_visible",
    "no_priority_signal"
)

df["action"] = np.where(
    df["score"] > 0,
    "refresh_review",
    "monitor"
)

queue = (
    df[
        [
            "content_id",
            "client_id",
            "score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        ["score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

print(queue.head(20))
print("\nRows:", len(queue))

              content_id          client_id  score        reason_code  \
0   content_8f39d99dfcfc  client_6208ef0f77  29981  stale_and_visible   
1   content_b3fccc13da09  client_6208ef0f77  29890  stale_and_visible   
2   content_88e1880bd3ab  client_19581e27de  29850  stale_and_visible   
3   content_9331e5f7eeab  client_19581e27de  29828  stale_and_visible   
4   content_83163890c43a  client_19581e27de  29822  stale_and_visible   
5   content_e859812ce999  client_19581e27de  29760  stale_and_visible   
6   content_afd71ac0a398  client_6208ef0f77  29751  stale_and_visible   
7   content_110a32997057  client_19581e27de  29717  stale_and_visible   
8   content_593e2753a70c  client_19581e27de  29708  stale_and_visible   
9   content_d636f1cf9880  client_6208ef0f77  29697  stale_and_visible   
10  content_00a44e45d37c  client_19581e27de  29692  stale_and_visible   
11  content_89e7611dd18c  client_6208ef0f77  29689  stale_and_visible   
12  content_53773e2a6df5  client_6208ef0f77  29654 

In [29]:
OUTPUT_PATH = "../../work/outputs/baseline_action_score.csv"

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(OUTPUT_PATH, index=False)

print("Wrote:", OUTPUT_PATH)

Wrote: ../../work/outputs/baseline_action_score.csv


In [30]:
OUTPUT_PATH = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(OUTPUT_PATH, index=False)

print("Wrote:", OUTPUT_PATH)

Wrote: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


In [31]:
print(queue["action"].value_counts())
print()
print(queue["reason_code"].value_counts())

action
monitor           23255
refresh_review     6745
Name: count, dtype: int64

reason_code
no_priority_signal    23255
stale_and_visible      6745
Name: count, dtype: int64


In [32]:
print(queue.head(10))

             content_id          client_id  score        reason_code  \
0  content_8f39d99dfcfc  client_6208ef0f77  29981  stale_and_visible   
1  content_b3fccc13da09  client_6208ef0f77  29890  stale_and_visible   
2  content_88e1880bd3ab  client_19581e27de  29850  stale_and_visible   
3  content_9331e5f7eeab  client_19581e27de  29828  stale_and_visible   
4  content_83163890c43a  client_19581e27de  29822  stale_and_visible   
5  content_e859812ce999  client_19581e27de  29760  stale_and_visible   
6  content_afd71ac0a398  client_6208ef0f77  29751  stale_and_visible   
7  content_110a32997057  client_19581e27de  29717  stale_and_visible   
8  content_593e2753a70c  client_19581e27de  29708  stale_and_visible   
9  content_d636f1cf9880  client_6208ef0f77  29697  stale_and_visible   

           action  rank  
0  refresh_review     1  
1  refresh_review     2  
2  refresh_review     3  
3  refresh_review     4  
4  refresh_review     5  
5  refresh_review     6  
6  refresh_review     7  

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10_ids = queue.head(10)["content_id"]

top10 = df[df["content_id"].isin(top10_ids)].copy()

top10 = (
    queue.head(10)[["rank", "content_id", "score", "reason_code", "action"]]
    .merge(
        top10[
            [
                "content_id",
                "days_since_last_update",
                "impressions_90d",
                "search_volume",
                "ctr",
                "avg_position",
                "content_type",
                "word_count"
            ]
        ],
        on="content_id",
        how="left"
    )
    .sort_values("rank")
)

print(top10.to_string(index=False))

 rank           content_id  score       reason_code         action  days_since_last_update  impressions_90d  search_volume  ctr  avg_position    content_type  word_count
    1 content_8f39d99dfcfc  29981 stale_and_visible refresh_review                     104            29981            0.0 0.05          34.3 keyword article      5748.0
    2 content_b3fccc13da09  29890 stale_and_visible refresh_review                     104            29890            0.0 0.03          40.2 keyword article      7384.0
    3 content_88e1880bd3ab  29850 stale_and_visible refresh_review                     104            29850           10.0 0.07           5.6 keyword article         NaN
    4 content_9331e5f7eeab  29828 stale_and_visible refresh_review                     104            29828          260.0 0.07           6.7 keyword article         NaN
    5 content_83163890c43a  29822 stale_and_visible refresh_review                     104            29822           40.0 0.44           3.1 keyword 

In [34]:
### Top-10 skeptical review

"""1. **Rank 1 — refresh_review.** It is 104 days stale and has 29,981 impressions, placing it near the top of the eligible visibility range. It could be wrong if the page is already performing well and does not need a content change.

2. **Rank 2 — refresh_review.** It is 104 days stale with 29,890 impressions, so the rule prioritizes its substantial search visibility. It could be wrong if the high impressions do not translate into a meaningful refresh opportunity.

3. **Rank 3 — refresh_review.** It is 104 days stale with 29,850 impressions and ranks highly because of its visibility. It could be wrong because its average position is already strong at 5.6, meaning a refresh might not be necessary.

4. **Rank 4 — refresh_review.** It is 104 days stale and has 29,828 impressions, giving it a high baseline score. It could be wrong if its strong average position of 6.7 means the existing content is already effective.

5. **Rank 5 — refresh_review.** It is 104 days stale with 29,822 impressions and a strong average position of 3.1. It could be wrong because the page already ranks very well, so changing it could create unnecessary risk.

6. **Rank 6 — refresh_review.** It is 104 days stale and has 29,760 impressions, making it highly visible within the rule's target zone. It could be wrong if the page's current performance is stable and a refresh would add little value.

7. **Rank 7 — refresh_review.** It is 104 days stale with 29,751 impressions, so it receives a high score from the visibility component. It could be wrong because its average position is relatively weak at 38.6, meaning the problem may require a different intervention than a simple content refresh.

8. **Rank 8 — refresh_review.** It is 104 days stale with 29,717 impressions and an average position of 1.5. It could be wrong because the page already ranks extremely well, making a refresh potentially unnecessary or risky.

9. **Rank 9 — refresh_review.** It is 104 days stale and has 29,708 impressions, which places it near the top of the queue. It could be wrong if the high visibility reflects stable performance rather than a need for updating.

10. **Rank 10 — refresh_review.** It is 104 days stale with 29,697 impressions and an average position of 36.3. It could be wrong because the poor ranking may require SEO or technical investigation rather than a content refresh."""

"1. **Rank 1 — refresh_review.** It is 104 days stale and has 29,981 impressions, placing it near the top of the eligible visibility range. It could be wrong if the page is already performing well and does not need a content change.\n\n2. **Rank 2 — refresh_review.** It is 104 days stale with 29,890 impressions, so the rule prioritizes its substantial search visibility. It could be wrong if the high impressions do not translate into a meaningful refresh opportunity.\n\n3. **Rank 3 — refresh_review.** It is 104 days stale with 29,850 impressions and ranks highly because of its visibility. It could be wrong because its average position is already strong at 5.6, meaning a refresh might not be necessary.\n\n4. **Rank 4 — refresh_review.** It is 104 days stale and has 29,828 impressions, giving it a high baseline score. It could be wrong if its strong average position of 6.7 means the existing content is already effective.\n\n5. **Rank 5 — refresh_review.** It is 104 days stale with 29,822 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
### Weak picks

"""The weakest part of this baseline is that it treats staleness and search visibility as sufficient
for a refresh recommendation. Several top-ranked pages already have strong average positions,
including pages at positions 1.5, 3.1, and 5.6. These pages may not benefit from a refresh even
though they satisfy the rule.

The rule also does not distinguish between a page that is highly visible but stable and one that
has an actual performance problem. This is a deliberate limitation of the simple baseline and is
one reason a later model should be tested against it."""

'The weakest part of this baseline is that it treats staleness and search visibility as sufficient\nfor a refresh recommendation. Several top-ranked pages already have strong average positions,\nincluding pages at positions 1.5, 3.1, and 5.6. These pages may not benefit from a refresh even\nthough they satisfy the rule.\n\nThe rule also does not distinguish between a page that is highly visible but stable and one that\nhas an actual performance problem. This is a deliberate limitation of the simple baseline and is\none reason a later model should be tested against it.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [36]:
### Self-check

"""- Two signals were audited using visible bucket tables with sample counts.
- Both signal verdicts were recorded as MIXED rather than forcing a positive conclusion.
- The rule uses only current/historical input features.
- `trend_direction` and `trend_pct` were not used as rule inputs.
- The rule has one score, one primary reason code, and an action label.
- The full 30,000-row queue was ranked.
- The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- The top 10 were reviewed skeptically, including a specific failure condition for each.
- The baseline is intentionally simple and provides a benchmark for the Week-5 model."""

'- Two signals were audited using visible bucket tables with sample counts.\n- Both signal verdicts were recorded as MIXED rather than forcing a positive conclusion.\n- The rule uses only current/historical input features.\n- `trend_direction` and `trend_pct` were not used as rule inputs.\n- The rule has one score, one primary reason code, and an action label.\n- The full 30,000-row queue was ranked.\n- The ranked queue is written to `work/outputs/baseline_action_score.csv`.\n- The top 10 were reviewed skeptically, including a specific failure condition for each.\n- The baseline is intentionally simple and provides a benchmark for the Week-5 model.'

In [37]:
print("Leakage check:")
print("trend_direction used in score:", "trend_direction" in str(df["score"].name))
print("Queue rows:", len(queue))
print("Unique content IDs:", queue["content_id"].nunique())
print("Output exists:", Path("/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv").exists())

Leakage check:
trend_direction used in score: False
Queue rows: 30000
Unique content IDs: 30000
Output exists: True


In [38]:
rule_features = [
    "days_since_last_update",
    "impressions_90d"
]

print("Actual rule features:", rule_features)

for col in ["trend_direction", "trend_pct", "is_declining_label"]:
    print(col, "used in rule:", col in rule_features)

Actual rule features: ['days_since_last_update', 'impressions_90d']
trend_direction used in rule: False
trend_pct used in rule: False
is_declining_label used in rule: False
